# Coding assistant generation tokens

In VS Code, select the Python kernel from this repository's `.venv/bin/python` and run all cells. Edit the next cell to select runs. Each run is read from `DATA_ROOT / "runs" / run_id`.

Total input includes cache reads and cache writes. The report includes usage from failed generation attempts when available. Copilot totals include all models recorded in the session, including search subagents.

In [1]:
from pathlib import Path

DATA_ROOT = Path("/Volumes/Extreme SSD/swebench-experiment-results/swebench-experiments")
RUN_IDS = ["codex-100", "copilot-100"]

In [2]:
import json
from collections import Counter, defaultdict

from IPython.display import Markdown, display


TOKEN_COLUMNS = ("Total input", "Cached input reads", "Cache writes", "Output")
USAGE_FIELDS = {
    "codex": ("input_tokens", "cached_input_tokens", "cache_write_input_tokens", "output_tokens"),
    "copilot": ("inputTokens", "cacheReadTokens", "cacheWriteTokens", "outputTokens"),
}


def parse_usage(assistant, usage):
    counts = Counter()
    for column, field in zip(TOKEN_COLUMNS, USAGE_FIELDS[assistant]):
        value = usage[field]
        if type(value) is not int or value < 0:
            raise ValueError(f"invalid token count for {field}")
        counts[column] = value
    return counts


def read_attempt(log_path, assistant):
    codex_counts = Counter()
    completed_turns = 0
    checkpoint = None

    with log_path.open(encoding="utf-8") as log:
        for line_number, line in enumerate(log, start=1):
            if not line.strip():
                continue
            try:
                event = json.loads(line)
                if assistant == "codex" and event.get("type") == "turn.completed":
                    codex_counts.update(parse_usage(assistant, event["usage"]))
                    completed_turns += 1
                elif assistant == "copilot" and event.get("type") == "session.usage_checkpoint":
                    checkpoint = event
            except (AttributeError, KeyError, TypeError, ValueError) as error:
                raise ValueError(f"line {line_number}: {error}") from error

    if assistant == "codex":
        if not completed_turns:
            raise ValueError("no turn.completed usage records")
        return codex_counts, {}

    if checkpoint is None:
        raise ValueError("no session.usage_checkpoint records")
    models = checkpoint["data"]["accountingSnapshot"]["modelMetrics"]
    if not isinstance(models, dict) or not models:
        raise ValueError("no model metrics in the final usage checkpoint")
    by_model = {
        model: parse_usage(assistant, metrics["usage"]) for model, metrics in models.items()
    }
    return sum(by_model.values(), Counter()), by_model


def read_run(run_dir):
    assistant = json.loads((run_dir / "run_metadata.json").read_text(encoding="utf-8"))["assistant"]
    if assistant not in USAGE_FIELDS:
        raise ValueError(f"unsupported assistant {assistant!r}")
    statuses = json.loads((run_dir / "generation_status.json").read_text(encoding="utf-8"))
    if not isinstance(statuses, list):
        raise ValueError("expected a list in generation_status.json")
    instance_ids = list(dict.fromkeys(attempt["instance_id"] for attempt in statuses))

    totals = Counter()
    by_model = defaultdict(Counter)
    counted = 0
    issues = []
    for instance_id in instance_ids:
        log_path = run_dir / instance_id / f"{assistant}.jsonl"
        try:
            counts, model_counts = read_attempt(log_path, assistant)
        except (OSError, KeyError, TypeError, ValueError) as error:
            issues.append(f"{log_path}: {error}")
            continue
        totals.update(counts)
        for model, counts in model_counts.items():
            by_model[model].update(counts)
        counted += 1

    return totals, by_model, counted, len(instance_ids), issues

In [3]:
def display_table(headers, rows):
    def row(values):
        return "| " + " | ".join(str(value).replace("|", r"\|") for value in values) + " |"

    display(Markdown("\n".join([row(headers), row(["---"] * len(headers)), *(row(values) for values in rows)])))


def format_counts(counts):
    return [f"{counts[column]:,}" if counts is not None else "—" for column in TOKEN_COLUMNS]


runs_dir = DATA_ROOT.expanduser() / "runs"
summary_rows = []
model_rows = []
issues = []
for run_id in dict.fromkeys(RUN_IDS):
    try:
        totals, by_model, counted, attempted, run_issues = read_run(runs_dir / run_id)
    except (OSError, KeyError, TypeError, ValueError) as error:
        summary_rows.append([run_id, "—", "Unavailable", *format_counts(None)])
        issues.append(f"{run_id}: {error}")
        continue

    status = "Unavailable" if not counted else ("Partial" if counted < attempted else "Complete")
    summary_rows.append([run_id, f"{counted}/{attempted}", status, *format_counts(totals if counted else None)])
    model_rows.extend([run_id, model, *format_counts(counts)] for model, counts in sorted(by_model.items()))
    issues.extend(run_issues)
    if not attempted:
        issues.append(f"{run_id}: generation_status.json lists no attempts")

display_table(["Run ID", "Attempts counted", "Coverage", *TOKEN_COLUMNS], summary_rows)
if model_rows:
    display(Markdown("### Copilot usage by model"))
    display_table(["Run ID", "Model", *TOKEN_COLUMNS], model_rows)
if issues:
    display(Markdown("### Data issues"))
    print("\n".join(issues))
if not summary_rows:
    print("Set RUN_IDS in the configuration cell to report generation usage.")

| Run ID | Attempts counted | Coverage | Total input | Cached input reads | Cache writes | Output |
| --- | --- | --- | --- | --- | --- | --- |
| codex-100 | 100/100 | Complete | 42,760,124 | 38,936,192 | 0 | 282,735 |
| copilot-100 | 100/100 | Complete | 51,832,720 | 48,952,703 | 2,563,075 | 489,150 |

### Copilot usage by model

| Run ID | Model | Total input | Cached input reads | Cache writes | Output |
| --- | --- | --- | --- | --- | --- |
| copilot-100 | copilot-search-a | 680,334 | 368,861 | 0 | 23,376 |
| copilot-100 | gpt-6.1-terra | 51,152,386 | 48,583,842 | 2,563,075 | 465,774 |